# Camada Ouro — Processo Sicredi

Pipeline da Camada Ouro lendo as tabelas da camada Silver (`workspace.silver`) e gravando no schema `workspace.gold`, conforme as regras do desafio:

- **`gold_transacoes_cartao`**: visão analítica única unindo cliente, conta, cartão e transação (base: `transacoes_financeiras`, sem transações estornadas).
- **`gold_resumo_cliente_mes`**: indicadores mensais por cliente (quantidade, valor total e ticket médio).
- **`gold_ranking_estabelecimento`**: ranking de estabelecimentos por valor total e quantidade de transações.
- **`gold_alertas_qualidade`**: resumo dos registros rejeitados nas quarentenas da Silver por entidade e tipo de erro.

Todas as tabelas Gold são reconstruídas de forma idempotente (overwrite) a cada execução do pipeline.

In [0]:
# ============================================================
# Parâmetros do pipeline Gold
# ============================================================
from pyspark.sql import functions as F, Window

CATALOG = "workspace"
SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"

def publicar_gold(df, tabela):
    """Grava o DataFrame na tabela Gold com overwrite (idempotente)."""
    saida = df.withColumn("timestamp_processamento", F.current_timestamp())
    saida.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela)
    n = saida.count()
    print(f"  Gold {tabela}: {n} registros")
    return n

# DataFrames base da Silver lidos uma única vez
transacoes_financeiras = spark.table(f"{SILVER}.transacoes_financeiras")
cartoes_silver = spark.table(f"{SILVER}.cartoes")
contas_silver = spark.table(f"{SILVER}.contas")
clientes_silver = spark.table(f"{SILVER}.clientes")

print("Parâmetros configurados:", SILVER, "->", GOLD)

In [0]:
# ============================================================
# 1. GOLD_TRANSACOES_CARTAO — visão analítica de cliente,
#    conta, cartão e transação em uma única tabela.
# Base: silver.transacoes_financeiras (SEM transações estornadas,
# para não contaminar os indicadores financeiros).
# ============================================================

# Join em cascata: transação -> cartão -> conta -> cliente
# A RI já foi garantida na Silver, então os joins nunca descartam linhas.
# --- Rename data_atualizacao columns to avoid conflicts ---
cartoes_ren = cartoes_silver.withColumnRenamed("data_atualizacao", "data_atualizacao_cartao")
contas_ren = contas_silver.withColumnRenamed("data_abertura", "data_abertura_conta")
clientes_ren = clientes_silver.withColumnRenamed("data_atualizacao", "data_atualizacao_cliente")

gold_transacoes = (
    transacoes_financeiras
    .join(cartoes_ren, "id_cartao", "inner")
    .join(contas_ren, "id_conta", "inner")
    .join(clientes_ren, "id_cliente", "inner")
)

gold_transacoes_analitico = gold_transacoes.select(
    # --- transação ---
    F.col("id_transacao"),
    F.col("data_transacao"),
    F.date_format("data_transacao", "yyyy-MM").alias("ano_mes"),
    F.col("valor"),
    F.col("nome_estabelecimento"),
    F.col("mcc"),
    F.col("cidade_estabelecimento"),
    # --- cartão ---
    F.col("id_cartao"),
    F.col("bandeira"),
    F.col("tipo_cartao"),
    F.col("status_cartao"),
    F.col("data_atualizacao_cartao"),
    # --- conta ---
    F.col("id_conta"),
    F.col("tipo_conta"),
    F.col("status_conta"),
    F.col("data_abertura_conta"),
    # --- cliente ---
    F.col("id_cliente"),
    F.col("nome").alias("nome_cliente"),
    F.col("cidade").alias("cidade_cliente"),
    F.col("estado").alias("estado_cliente"),
    F.col("data_atualizacao_cliente"),
)

qtd_gt = publicar_gold(gold_transacoes_analitico, f"{GOLD}.gold_transacoes_cartao")
print(f"GOLD_TRANSACOES_CARTAO: {qtd_gt} transações (igual a silver.transacoes_financeiras)")
display(gold_transacoes_analitico.limit(10))

In [0]:
# ============================================================
# 2. GOLD_RESUMO_CLIENTE_MES — indicadores mensais por cliente.
# Agregação sobre a visão analítica gold_transacoes_cartao
# (apenas transações não estornadas).
# ============================================================
base_cliente_mes = spark.table(f"{GOLD}.gold_transacoes_cartao")

resumo_cliente_mes = (
    base_cliente_mes
    .groupBy("id_cliente", "nome_cliente", "estado_cliente", "ano_mes")
    .agg(
        F.count("id_transacao").alias("qtd_transacoes"),
        F.sum("valor").alias("valor_total"),
        F.round(F.avg("valor"), 2).alias("ticket_medio"),
        F.min("data_transacao").alias("primeira_transacao"),
        F.max("data_transacao").alias("ultima_transacao"),
    )
    .orderBy("id_cliente", "ano_mes")
)

qtd_rcm = publicar_gold(resumo_cliente_mes, f"{GOLD}.gold_resumo_cliente_mes")
print(f"GOLD_RESUMO_CLIENTE_MES: {qtd_rcm} combinações cliente x mês")
display(resumo_cliente_mes.limit(10))

In [0]:
# ============================================================
# 3. GOLD_RANKING_ESTABELECIMENTO — ranking de estabelecimentos
# por valor total e quantidade de transações (exclui estornadas).
# ============================================================
base_ranking = spark.table(f"{GOLD}.gold_transacoes_cartao")

agregado_estab = (
    base_ranking
    .groupBy("nome_estabelecimento", "mcc")
    .agg(
        F.count("id_transacao").alias("qtd_transacoes"),
        F.sum("valor").alias("valor_total"),
        F.round(F.avg("valor"), 2).alias("ticket_medio"),
        F.countDistinct("id_cliente").alias("qtd_clientes_distintos"),
    )
)

# Ranking por valor (1 = maior valor) e por quantidade (1 = mais transações)
# partitionBy(lit(1)): janela global única — evita warning de janela sem partição
w_valor = Window.partitionBy(F.lit(1)).orderBy(F.col("valor_total").desc(), F.col("qtd_transacoes").desc())
w_qtd = Window.partitionBy(F.lit(1)).orderBy(F.col("qtd_transacoes").desc(), F.col("valor_total").desc())

ranking_estabelecimento = (
    agregado_estab
    .withColumn("ranking_valor", F.row_number().over(w_valor))
    .withColumn("ranking_quantidade", F.row_number().over(w_qtd))
    .select(
        "ranking_valor",
        "ranking_quantidade",
        "nome_estabelecimento",
        "mcc",
        "qtd_transacoes",
        "valor_total",
        "ticket_medio",
        "qtd_clientes_distintos",
    )
    .orderBy("ranking_valor")
)

qtd_re = publicar_gold(ranking_estabelecimento, f"{GOLD}.gold_ranking_estabelecimento")
print(f"GOLD_RANKING_ESTABELECIMENTO: {qtd_re} estabelecimentos (par nome x mcc)")
display(ranking_estabelecimento.limit(10))

In [0]:
# ============================================================
# 4. GOLD_ALERTAS_QUALIDADE — resumo dos registros rejeitados
# (quarentenas da Silver) por entidade e tipo de erro.
# O motivo_erro pode conter vários erros combinados ("A | B");
# explode para contar cada tipo de erro individualmente.
# ============================================================
quarentenas = [
    ("clientes", f"{SILVER}.quarentena_clientes"),
    ("contas", f"{SILVER}.quarentena_contas"),
    ("cartoes", f"{SILVER}.quarentena_cartoes"),
    ("transacoes_cartao", f"{SILVER}.quarentena_transacoes_cartao"),
    ("estornos", f"{SILVER}.quarentena_estornos"),
]

alertas = []
for entidade, t_quar in quarentenas:
    m = (
        spark.table(t_quar)
        # separa motivos combinados em linhas individuais
        .withColumn("tipo_erro", F.explode(F.split(F.col("motivo_erro"), " \\| ")))
        .groupBy("tipo_erro")
        .agg(F.count("*").alias("qtd_registros"))
        .withColumn("entidade", F.lit(entidade))
        .select("entidade", "tipo_erro", "qtd_registros")
    )
    alertas.append(m)

df_alertas = alertas[0]
for m in alertas[1:]:
    df_alertas = df_alertas.unionByName(m)

alertas_qualidade = (
    df_alertas
    .groupBy("entidade", "tipo_erro")
    .agg(F.sum("qtd_registros").alias("qtd_registros"))
    .orderBy("entidade", F.col("qtd_registros").desc())
)

qtd_aq = publicar_gold(alertas_qualidade, f"{GOLD}.gold_alertas_qualidade")
print(f"GOLD_ALERTAS_QUALIDADE: {qtd_aq} combinações entidade x tipo_erro")
display(alertas_qualidade)

In [0]:
# ============================================================
# 5. RESUMO DO PIPELINE GOLD + VALIDAÇÕES DE CONSISTÊNCIA
# ============================================================
tabelas_gold = [
    ("gold_transacoes_cartao", f"{GOLD}.gold_transacoes_cartao"),
    ("gold_resumo_cliente_mes", f"{GOLD}.gold_resumo_cliente_mes"),
    ("gold_ranking_estabelecimento", f"{GOLD}.gold_ranking_estabelecimento"),
    ("gold_alertas_qualidade", f"{GOLD}.gold_alertas_qualidade"),
]

linhas = [(nome, spark.table(t).count()) for nome, t in tabelas_gold]
display(spark.createDataFrame(linhas, "tabela string, registros long"))

# --- Validações de consistência ---
gold_trans = spark.table(f"{GOLD}.gold_transacoes_cartao")
silver_fin = spark.table(f"{SILVER}.transacoes_financeiras")

print("1. gold_transacoes == silver.transacoes_financeiras:",
      gold_trans.count(), "vs", silver_fin.count())

soma_gold = gold_trans.agg(F.sum("valor").alias("v")).collect()[0]["v"]
soma_silver = silver_fin.agg(F.sum("valor").alias("v")).collect()[0]["v"]
print("2. soma dos valores gold == silver:", float(soma_gold), "vs", float(soma_silver))

soma_resumo = spark.table(f"{GOLD}.gold_resumo_cliente_mes") \
    .agg(F.sum("valor_total").alias("v")).collect()[0]["v"]
print("3. soma do resumo cliente-mês == soma gold:", float(soma_resumo), "vs", float(soma_gold))

soma_ranking = spark.table(f"{GOLD}.gold_ranking_estabelecimento") \
    .agg(F.sum("valor_total").alias("v")).collect()[0]["v"]
print("4. soma do ranking == soma gold:", float(soma_ranking), "vs", float(soma_gold))

total_rejeitados_silver = 0
for _, t in [("clientes", f"{SILVER}.quarentena_clientes"), ("contas", f"{SILVER}.quarentena_contas"),
             ("cartoes", f"{SILVER}.quarentena_cartoes"), ("transacoes", f"{SILVER}.quarentena_transacoes_cartao"),
             ("estornos", f"{SILVER}.quarentena_estornos")]:
    total_rejeitados_silver += spark.table(t).count()
total_alertas = spark.table(f"{GOLD}.gold_alertas_qualidade").agg(F.sum("qtd_registros")).collect()[0][0]
print("5. total alertas gold >= registros rejeitados na silver (pode ser maior:",
      "motivos combinados contam em cada tipo):", total_alertas, "vs", total_rejeitados_silver)

print("\nPipeline Gold concluído com sucesso.")